In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import patsy
from statsmodels.stats.outliers_influence import variance_inflation_factor

path = r'C:\Users\13676\Desktop\senna_cohort.csv'
df = pd.read_csv(path, encoding='gb18030',
                 encoding_errors='replace', low_memory=False)

time_cols = ['index_time', 'baseline_k_charttime', 'hypokalemia_time',
             'window_event_time', 'admittime', 'dischtime', 'deathtime',
             'baseline_egfr_charttime']
for c in time_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')

df_main = df.dropna(subset=['senna_exposed', 'hypokalemia_event',
                             'baseline_k', 'age']).copy()
df_main['y'] = df_main['hypokalemia_event'].astype(int)
df_main['senna_exposed'] = df_main['senna_exposed'].astype(int)
df_main['gender_male'] = (df_main['gender'].str.upper().str[0] == 'M').astype(int)

print('df_main 行数:', len(df_main))
print('暴露组:', (df_main['senna_exposed']==1).sum())
print('低钾事件:', df_main['y'].sum())

df_main 行数: 41529
暴露组: 24226
低钾事件: 8752


In [2]:
continuous_vars = [
    'age', 'baseline_k', 'baseline_creatinine', 'baseline_egfr',
    'drug_count_no_laxatives',
]
binary_vars = [
    'gender_male',
    'chf', 'ckd', 'dm',
    'diuretic', 'acei_arb', 'digoxin', 'insulin', 'statin', 'anticoagulant',
    'lactulose', 'peg', 'bisacodyl', 'docusate',
    'potassium_supplement', 'ppi', 'beta2_agonist',
    'corticosteroid', 'diarrhea_vomiting',
]
age_group_levels = ['65-74', '75-84', '85+']
print('清单 OK')

清单 OK


In [3]:
# ========== 单元格 3：主动对照分析 ==========
# 排除 senna 使用者，在剩余样本中单独检验其他泻药
df_no_senna = df_main[df_main['senna_exposed'] == 0].copy()
n_no_lax = int((df_no_senna[['lactulose','peg','bisacodyl','docusate']]
                .sum(axis=1) == 0).sum())
print(f'排除 senna 后样本量: {len(df_no_senna)}')
print(f'  低钾率: {df_no_senna["y"].mean()*100:.2f}%')
print(f'  无任何泻药者: {n_no_lax} '
      f'({n_no_lax/len(df_no_senna)*100:.1f}%)')

def active_control_model(df, lax_var):
    """在无 senna 队列中，检验单个其他泻药与低钾的关联"""
    formula = (
        f'y ~ {lax_var} + gender_male + '
        'C(age_group, Treatment("65-74")) + '
        'baseline_k + drug_count_no_laxatives + '
        'chf + ckd + dm + '
        'diuretic + acei_arb + digoxin + insulin + statin + anticoagulant + '
        'ppi + beta2_agonist + corticosteroid'
    )
    m = smf.logit(formula, data=df).fit(disp=0)
    or_ = np.exp(m.params[lax_var])
    lo  = np.exp(m.conf_int().loc[lax_var, 0])
    hi  = np.exp(m.conf_int().loc[lax_var, 1])
    p   = m.pvalues[lax_var]
    return or_, lo, hi, p, int(df[lax_var].sum()), len(df)

results = []
lax_list = [
    ('Senna (main analysis)', 'senna_exposed', df_main),
    ('Lactulose',             'lactulose',      df_no_senna),
    ('PEG',                   'peg',            df_no_senna),
    ('Bisacodyl',             'bisacodyl',      df_no_senna),
    ('Docusate',              'docusate',       df_no_senna),
]
for name, var, data in lax_list:
    try:
        or_, lo, hi, p, n_exp, n_tot = active_control_model(data, var)
        results.append({'Laxative': name,
                        'N exposed': n_exp,
                        'N total': n_tot,
                        'aOR': round(or_, 3),
                        '95% CI': f'{lo:.3f}–{hi:.3f}',
                        'P': f'{p:.3g}'})
    except Exception as e:
        results.append({'Laxative': name, 'N exposed': 'ERR',
                        'N total': 'ERR', 'aOR': np.nan,
                        '95% CI': 'ERR', 'P': str(e)[:30]})

table_active = pd.DataFrame(results)
print('\n--- 主动对照分析：各泻药与低钾血症关联 ---')
print(table_active.to_string(index=False))
table_active.to_excel('TableS_active_comparator.xlsx', index=False)
print('\n已保存 TableS_active_comparator.xlsx')

排除 senna 后样本量: 17303
  低钾率: 18.82%
  无任何泻药者: 9339 (54.0%)

--- 主动对照分析：各泻药与低钾血症关联 ---
             Laxative  N exposed  N total   aOR      95% CI        P
Senna (main analysis)      24226    41529 1.207 1.145–1.272 2.02e-12
            Lactulose        991    17303 1.375 1.170–1.615 0.000109
                  PEG       1873    17303 0.887 0.778–1.012   0.0748
            Bisacodyl       3772    17303 0.929 0.835–1.033    0.174
             Docusate       6273    17303 0.760 0.693–0.833 5.25e-09

已保存 TableS_active_comparator.xlsx


In [6]:
# ========== 单元格 4：剂量-反应分析（修复版）==========
exp = df_main[df_main['senna_exposed'] == 1].copy()

for v in ['senna_total_dose', 'senna_days', 'senna_order_count']:
    exp[v] = pd.to_numeric(exp[v], errors='coerce')

print(f'暴露组样本量: {len(exp)}')
print(f'  低钾事件: {exp["y"].sum()} ({exp["y"].mean()*100:.2f}%)')

def dose_response(exp, var, label):
    print(f'\n--- {label}（三分位）---')
    df_ = exp[[var, 'y', 'gender_male', 'baseline_k',
               'drug_count_no_laxatives', 'chf', 'diuretic', 'insulin']].dropna()

    # ---------- 稳健分箱：处理离散变量 ----------
    q1, q2 = df_[var].quantile([1/3, 2/3])
    print(f'  原始切点: Q1={q1:.2f}, Q2={q2:.2f}')

    unique_vals = df_[var].nunique()
    # 若 Q1 == Q2（离散或零膨胀），改用唯一值分箱
    if q1 == q2 or unique_vals <= 3:
        print(f'  ⚠️ Q1==Q2 或唯一值太少（{unique_vals}），改用唯一值分箱')
        cats = sorted(df_[var].unique())
        if len(cats) == 1:
            print(f'  该变量只有 1 个唯一值，跳过'); return None
        # 按唯一值三等分（或直接二分类）
        n_cat = min(3, len(cats))
        cuts = np.linspace(0, len(cats), n_cat + 1).round().astype(int)
        bins_use = [-np.inf]
        for i in range(1, n_cat):
            bins_use.append(cats[cuts[i] - 1] + 0.5)
        bins_use.append(np.inf)
        labels_use = ['Low', 'Medium', 'High'][:n_cat] if n_cat == 3 else ['Low', 'High']
        df_['cat'] = pd.cut(df_[var], bins=bins_use, labels=labels_use)
    else:
        df_['cat'] = pd.cut(df_[var],
                            bins=[-np.inf, q1, q2, np.inf],
                            labels=['Low', 'Medium', 'High'])

    print(f'  各组例数: {df_["cat"].value_counts().sort_index().to_dict()}')

    # 各组低钾率
    rate = df_.groupby('cat', observed=True)['y'].agg(['sum', 'count', 'mean'])
    print(f'  各组低钾率:')
    for cat in rate.index:
        r = rate.loc[cat]
        print(f'    {cat}: {int(r["sum"])}/{int(r["count"])} '
              f'({r["mean"]*100:.2f}%)')

    # 回归：以第一个类别为参照
    ref_cat = df_['cat'].cat.categories[0]
    formula = (
        f'y ~ C(cat, Treatment("{ref_cat}")) + gender_male + baseline_k + '
        'drug_count_no_laxatives + chf + diuretic + insulin'
    )
    m = smf.logit(formula, data=df_).fit(disp=0)
    for term in m.params.index:
        if 'C(cat' in term:
            lv = term.split('T.')[1].rstrip(']')
            or_ = np.exp(m.params[term])
            lo  = np.exp(m.conf_int().loc[term, 0])
            hi  = np.exp(m.conf_int().loc[term, 1])
            p   = m.pvalues[term]
            print(f'  {lv} vs {ref_cat}: aOR = {or_:.3f} '
                  f'({lo:.3f}–{hi:.3f}), P = {p:.3g}')

    # 趋势检验
    df_['cat_num'] = df_['cat'].cat.codes + 1
    formula_trend = (
        'y ~ cat_num + gender_male + baseline_k + '
        'drug_count_no_laxatives + chf + diuretic + insulin'
    )
    mt = smf.logit(formula_trend, data=df_).fit(disp=0)
    or_t = np.exp(mt.params['cat_num'])
    lo_t = np.exp(mt.conf_int().loc['cat_num', 0])
    hi_t = np.exp(mt.conf_int().loc['cat_num', 1])
    p_t  = mt.pvalues['cat_num']
    print(f'  P for trend = {p_t:.4g} '
          f'(每升高一个三分位，aOR = {or_t:.3f}, '
          f'95% CI {lo_t:.3f}–{hi_t:.3f})')
    return df_

df_dose1 = dose_response(exp, 'senna_total_dose',  '累积剂量 (senna_total_dose)')
df_dose2 = dose_response(exp, 'senna_days',        '用药天数 (senna_days)')
df_dose3 = dose_response(exp, 'senna_order_count', '医嘱次数 (senna_order_count)')

暴露组样本量: 24226
  低钾事件: 5495 (22.68%)

--- 累积剂量 (senna_total_dose)（三分位）---
  原始切点: Q1=3.00, Q2=17.20
  各组例数: {'Low': 8125, 'Medium': 12361, 'High': 3740}
  各组低钾率:
    Low: 1706/8125 (21.00%)
    Medium: 2505/12361 (20.27%)
    High: 1284/3740 (34.33%)
  Medium vs Low: aOR = 1.020 (0.949–1.097), P = 0.592
  High vs Low: aOR = 1.641 (1.495–1.802), P = 2.75e-25
  P for trend = 2.268e-19 (每升高一个三分位，aOR = 1.239, 95% CI 1.183–1.299)

--- 用药天数 (senna_days)（三分位）---
  原始切点: Q1=0.00, Q2=0.00
  ⚠️ Q1==Q2 或唯一值太少（664），改用唯一值分箱
  各组例数: {'Low': 23406, 'Medium': 566, 'High': 254}
  各组低钾率:
    Low: 4986/23406 (21.30%)
    Medium: 327/566 (57.77%)
    High: 182/254 (71.65%)
  Medium vs Low: aOR = 3.090 (2.572–3.712), P = 2.06e-33
  High vs Low: aOR = 3.976 (2.940–5.376), P = 3.14e-19
  P for trend = 2.452e-43 (每升高一个三分位，aOR = 2.393, 95% CI 2.114–2.709)

--- 医嘱次数 (senna_order_count)（三分位）---
  原始切点: Q1=1.00, Q2=2.00
  各组例数: {'Low': 15572, 'Medium': 5904, 'High': 2750}
  各组低钾率:
    Low: 2885/15572 (18.53%)
    